# 05 · RadioVue: one X-ray in, synthetic view + CT out

> **Research use only.** Generated radiographs and CT volumes are synthetic and patient-conditioned. They are
> estimates, not acquisitions of the patient and not clinical ground truth. Never use them for diagnosis or treatment.
> Upload only images you are authorised to use, and remove burned-in identifiers first.

**How to use:** `Runtime > Change runtime type > T4 GPU` (or better), then `Runtime > Run all`. After setup
(~10 min on first run), the last cell prints a **public Gradio link**. Open it, upload one chest X-ray,
optionally state its view, and click **Run**. No other cells need to be touched.

What happens on Run: view identification → (optional) MedGemma anatomy → segmentation → Parallax complementary
view (4 seeds + uncertainty) → QC gate → Volumea CT → display and download. A synthetic view that fails QC is
never sent to Volumea, and lateral inputs stop before synthesis (no validated lateral→frontal generator exists).

In [ ]:
#@title 1. Configuration { display-mode: "form" }
REPO_URL = "https://github.com/sanafathima-00/RadioVue.git"  #@param {type:"string"}
REPO_BRANCH = "claude/single-xray-vlm-pipeline"  #@param {type:"string"}
#@markdown If the pipeline code is not pushed yet, upload `radiovue_colab_bundle.zip` (see SETUP_GUIDE.md) to this Drive path:
CODE_BUNDLE_ON_DRIVE = "RadioVue/code/radiovue_colab_bundle.zip"  #@param {type:"string"}
#@markdown Mount Google Drive to persist outputs (Google asks you to approve access once). Models download from Hugging Face if not on Drive.
USE_DRIVE = True  #@param {type:"boolean"}
DRIVE_ROOT = "/content/drive/MyDrive/RadioVue"  #@param {type:"string"}
DVG_CHECKPOINTS_ON_DRIVE = DRIVE_ROOT + "/Volumea/models/DVG-Diffusion" if USE_DRIVE else None
PARALLAX_MODEL_ON_DRIVE = DRIVE_ROOT + "/Parallax/models/DiT-fb-512" if USE_DRIVE else None
OUTPUT_ROOT = DRIVE_ROOT + "/outputs/single_xray_pipeline" if USE_DRIVE else "/content/radiovue_outputs"
REPO_DIR = "/content/RadioVue"

In [ ]:
#@title 2. GPU, Drive and code
import os, sys, subprocess, zipfile, importlib.util
from pathlib import Path

smi = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
print(smi.stdout[:700] or smi.stderr)
assert smi.returncode == 0, "No GPU. Runtime > Change runtime type > GPU (L4/A100 recommended, T4 minimum)."

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

bundle = Path("/content/drive/MyDrive") / CODE_BUNDLE_ON_DRIVE
if not Path(REPO_DIR, "radiovue_pipeline").exists():
    if USE_DRIVE and bundle.is_file():
        Path(REPO_DIR).mkdir(parents=True, exist_ok=True)
        zipfile.ZipFile(bundle).extractall(REPO_DIR)
        print("Code from Drive bundle:", bundle)
    else:
        subprocess.run(["git", "clone", "--depth", "1", "-b", REPO_BRANCH, REPO_URL, REPO_DIR], check=True)
assert Path(REPO_DIR, "radiovue_pipeline").exists(), (
    "radiovue_pipeline/ not found. Push the branch that contains it, or upload the code bundle (SETUP_GUIDE.md).")
os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
rev = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
print("Code revision:", rev or "(bundle, see BUNDLE_MANIFEST.json)")

# colab_setup has no third-party imports, so it can run before packages are installed.
spec = importlib.util.spec_from_file_location("colab_setup", f"{REPO_DIR}/radiovue_pipeline/colab_setup.py")
cs = importlib.util.module_from_spec(spec); spec.loader.exec_module(cs)
GPU = cs.gpu_report()

In [ ]:
#@title 3. Install dependencies and stage models (first run ~10-15 min)
cs.install_runtime(parallax=True, dvg=True, tools=True, validation=False, app=True)
cs.build_drr_projector(f"{REPO_DIR}/Volumea")
DVG_CKPT_DIR = cs.stage_dvg_checkpoints(DVG_CHECKPOINTS_ON_DRIVE, verify_hash=False)
cs.ensure_parallax_shims(f"{REPO_DIR}/Parallax")
PARALLAX_MODEL_DIR = cs.stage_parallax_model(PARALLAX_MODEL_ON_DRIVE)
HF_TOKEN = cs.hf_token_from_colab_secrets()  # value is never printed
assert HF_TOKEN, "Add HF_TOKEN in Colab Secrets (key icon) and enable notebook access."

In [ ]:
#@title 4. Start the app (prints a public link; keep this cell running)
import json
from radiovue_pipeline import PipelineConfig
from radiovue_pipeline.pipeline import Components
from radiovue_pipeline.app import build_app
cfg = PipelineConfig(output_root=OUTPUT_ROOT, volumea_root=f"{REPO_DIR}/Volumea", parallax_root=f"{REPO_DIR}/Parallax",
                     dvg_checkpoint_dir=DVG_CKPT_DIR, parallax_model_dir=PARALLAX_MODEL_DIR,
                     run_medgemma=MEDGEMMA_PYTHON is not None,
                     free_gpu_between_stages=MEDGEMMA_PYTHON is not None)  # T4: only release models when MedGemma also needs VRAM
calib_path = Path(OUTPUT_ROOT) / "calibration" / "geometry_calibration.json"
if calib_path.exists():
    calib = json.loads(calib_path.read_text())
    for key in ("lateral_azimuth_deg", "lateral_azimuth_calibrated", "parallax_output_transform",
                "dvg_frontal_transform", "dvg_lateral_transform", "dvg_geometry_calibrated"):
        setattr(cfg, key, calib[key])
    print("Loaded geometry calibration from notebook 03")
Path(OUTPUT_ROOT).mkdir(parents=True, exist_ok=True)
comps = Components.load(cfg, medgemma_python=MEDGEMMA_PYTHON, repo_root=REPO_DIR)
build_app(cfg, comps).queue().launch(share=True, debug=True)